In [ ]:
from typing import TypedDict , Optional
from langgraph.graph import StateGraph , START , END
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
client = OpenAI(api_key=my_api_key)
#print(f'my api key is {my_api_key}')

In [5]:
client = OpenAI(api_key=my_api_key)


# --- Shared State ---
class LibraryState(TypedDict):
    question: Optional[str]
    faq_flag: Optional[bool]
    faq_answer: Optional[str]
    checkout_flag: Optional[bool]
    checkout_answer: Optional[str]
    final_answer: Optional[str]


def ClassifierAgent(state: LibraryState):
    print(f"ClassifierAgent Initial state: {state}")

    question = state["question"]

    messages = [
        {
            "role": "system",
            "content": (
                "You are a classifier agent in a library system.\n"
                "Determine whether the question involves:\n"
                "1. Library FAQs (hours, rules, membership)\n"
                "2. Book availability or checkout\n\n"
                "Return ONLY valid JSON:\n"
                "{\n"
                '  "faq_flag": true | false,\n'
                '  "checkout_flag": true | false\n'
                "}"
            )
        },
        {"role": "user", "content": question}
    ]

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
    )

    import json
    parsed = json.loads(response.choices[0].message.content)

    return {
        "faq_flag": bool(parsed.get("faq_flag", False)),
        "checkout_flag": bool(parsed.get("checkout_flag", False)),
    }


def FAQAgent(state: LibraryState):
    print(f"FAQAgent state: {state}")

    if not state.get("faq_flag"):
        return {"faq_answer": None}

    messages = [
        {"role": "system", "content": "You answer library FAQ questions."},
        {"role": "user", "content": state["question"]}
    ]

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
    )

    return {
        "faq_answer": response.choices[0].message.content
    }



def CheckoutAgent(state: LibraryState):
    print(f"CheckoutAgent state: {state}")

    if not state.get("checkout_flag"):
        return {"checkout_answer": None}

    # Connect to library API and fetch real data here. 
    # Once checkout is complete, call LLM to generate a response based 
    # on the API result.
    
    # For demo, we use LLM to generate a response.
    messages = [
        {"role": "system", "content": "You answer questions about book availability and checkout."},
        {"role": "user", "content": state["question"]}
    ]

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,
    )

    return {
        "checkout_answer": response.choices[0].message.content
    }


def ResponseAgent(state: LibraryState):
    print(f"ResponseAgent state: {state}")

    faq = state.get("faq_answer")
    checkout = state.get("checkout_answer")

    parts = []
    if faq:
        parts.append(f"FAQ Information:\n{faq}")
    if checkout:
        parts.append(f"Checkout Information:\n{checkout}")

    combined_context = "\n\n".join(parts)

    messages = [
        {
            "role": "system",
            "content": "Combine the information into a helpful response."
        },
        {
            "role": "user",
            "content": f"Question: {state['question']}\n\n{combined_context}"
        }
    ]

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,
    )

    return {
        "final_answer": response.choices[0].message.content
    }


# --- Build the Graph ---
builder = StateGraph(LibraryState)
builder.add_node("ClassifierAgent", ClassifierAgent)
builder.add_node("FAQAgent", FAQAgent)
builder.add_node("CheckoutAgent", CheckoutAgent)
builder.add_node("ResponseAgent", ResponseAgent)

builder.add_edge(START, "ClassifierAgent")
builder.add_edge("ClassifierAgent", "FAQAgent")
builder.add_edge("ClassifierAgent", "CheckoutAgent")
builder.add_edge("FAQAgent", "ResponseAgent")
builder.add_edge("CheckoutAgent", "ResponseAgent")
builder.add_edge("ResponseAgent", END)

graph = builder.compile()

In [6]:
user_input = input("Ask a question about the library: ")
result = graph.invoke({"question": user_input})
print("\n--- Final Answer ---")
print(result["final_answer"])

ClassifierAgent Initial state: {'question': 'what time liberary opens'}
CheckoutAgent state: {'question': 'what time liberary opens', 'faq_flag': True, 'checkout_flag': False}FAQAgent state: {'question': 'what time liberary opens', 'faq_flag': True, 'checkout_flag': False}

ResponseAgent state: {'question': 'what time liberary opens', 'faq_flag': True, 'faq_answer': "Please specify the name or location of the library you're inquiring about, so I can provide the accurate opening hours.", 'checkout_flag': False, 'checkout_answer': None}

--- Final Answer ---
I can help with that, but I need the library’s name or location. Which library are you asking about? Please provide:

- The library name (e.g., Central Library) or
- The city/branch location (e.g., Springfield, Main Street Branch)

If you’re not sure of the name, share the address or city and I’ll look up the hours. Note that hours can vary by day, holidays, and special events. Once you provide the details, I’ll give you the current 

In [7]:
# --- Run ---
result = graph.invoke({"question": 
                       '''When does Cupertino library open? I would like to check out a book as 
                       well.'''})
print("\n--- Final Answer ---")
print(result["final_answer"])

ClassifierAgent Initial state: {'question': 'When does Cupertino library open? I would like to check out a book as \n                       well.'}
CheckoutAgent state: {'question': 'When does Cupertino library open? I would like to check out a book as \n                       well.', 'faq_flag': True, 'checkout_flag': True}FAQAgent state: {'question': 'When does Cupertino library open? I would like to check out a book as \n                       well.', 'faq_flag': True, 'checkout_flag': True}

ResponseAgent state: {'question': 'When does Cupertino library open? I would like to check out a book as \n                       well.', 'faq_flag': True, 'faq_answer': "Cupertino Library typically opens at 10:00 AM Monday through Saturday and is closed on Sundays. However, hours may vary on holidays or due to special events. I recommend checking the Cupertino Library's official website or contacting them directly at (408) 861-6454 for the most current hours and information on checking out a b

In [8]:
result = graph.invoke({"question": "Is The Hobbit available? When does the library open"})
print("\n--- Final Answer ---")
print(result["final_answer"])

ClassifierAgent Initial state: {'question': 'Is The Hobbit available? When does the library open'}
CheckoutAgent state: {'question': 'Is The Hobbit available? When does the library open', 'faq_flag': True, 'checkout_flag': True}FAQAgent state: {'question': 'Is The Hobbit available? When does the library open', 'faq_flag': True, 'checkout_flag': True}

ResponseAgent state: {'question': 'Is The Hobbit available? When does the library open', 'faq_flag': True, 'faq_answer': 'Please check with your local library to see if The Hobbit is currently available in their collection. Library hours vary by location, so I recommend visiting their website or contacting them directly for their opening hours.', 'checkout_flag': True, 'checkout_answer': 'I can help, but I need to know your library branch to check real-time availability and hours.\n\nPlease tell me:\n- Which library or branch (city) you’re using\n- Whether you want The Hobbit in a specific format (physical book, eBook, audiobook) or all f